# 📐 Lab 02-1: Linear Regression with TensorFlow
### TensorFlow 1.x 스타일로 배우는 선형 회귀

---

이 노트북은 **TensorFlow의 핵심 동작 원리**를 이해하기 위해  
일부러 TF1 스타일(`Session`, `Variable`, `Graph`)로 작성합니다.

> 💡 TF2에서도 `tf.compat.v1`을 통해 동일하게 실행됩니다.

### 핵심 개념
| 개념 | 설명 |
|------|------|
| **Graph** | 연산의 설계도 — 실행 전까지는 값이 없음 |
| **Session** | Graph를 실제로 실행하는 엔진 |
| **Variable** | 학습을 통해 값이 바뀌는 파라미터 (W, b) |
| **Tensor** | 다차원 배열 — TF의 기본 데이터 단위 |
| **Optimizer** | cost를 줄이는 방향으로 W, b를 업데이트 |


---
## 1. 라이브러리 설치 및 버전 확인


In [ ]:
# TensorFlow 설치 (Codespace에서 처음 실행할 때)
# !pip install tensorflow

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow 버전: {tf.__version__}")
print("TF1 compat 모드 활성화 ✓")


---
## 2. 선형 회귀란?

**목표**: 데이터를 가장 잘 설명하는 직선 `y = Wx + b` 를 찾는 것

```
y = W·x + b
    ↑       ↑
  기울기   절편 (bias)
```

### 우리가 찾을 값
- 데이터: x = [1, 2, 3],  y = [1, 2, 3]
- 정답:   **W = 1.0,  b = 0.0** (y = x 직선)
- 하지만 TF는 이걸 **모르는 채로** 학습을 통해 스스로 찾습니다

### Cost Function (비용 함수)
예측값과 실제값의 차이를 측정합니다:

$$\text{cost}(W, b) = \frac{1}{m}\sum_{i=1}^{m}(H(x_i) - y_i)^2$$

cost가 작을수록 좋은 모델입니다.


---
## 3. TensorFlow Graph 설계

TF1에서는 코드를 **두 단계**로 나눕니다:

```
1단계: Graph 설계  →  "어떻게 계산할지 설계도 그리기"
2단계: Session 실행  →  "설계도대로 실제 계산 실행"
```

아래 코드는 **아직 실행이 아닙니다** — 설계도를 그리는 단계입니다.


In [ ]:
# 재현성을 위한 랜덤 시드 고정
tf.set_random_seed(777)

# ── 데이터 정의 ─────────────────────────────────
x_train = [1, 2, 3]
y_train = [1, 2, 3]

# ── Variable: 학습 가능한 파라미터 ──────────────
# tf.random_normal([1]) : 평균 0, 표준편차 1인 난수로 초기화
# W, b는 학습 중에 계속 업데이트됨
W = tf.Variable(tf.random_normal([1]), name="weight")
b = tf.Variable(tf.random_normal([1]), name="bias")

print("W:", W)   # 아직 값이 없음 — Tensor 객체
print("b:", b)   # 아직 값이 없음 — Tensor 객체
print()
print("⚠️  아직 실제 값이 없습니다 — Graph(설계도)만 만들어진 상태")


In [ ]:
# ── Hypothesis (가설) ───────────────────────────
# H(x) = W * x + b
hypothesis = x_train * W + b

# ── Cost Function ────────────────────────────────
# MSE (Mean Squared Error)
cost = tf.reduce_mean(tf.square(hypothesis - y_train))
#      ↑ 평균         ↑ 제곱        ↑ 차이

# ── Optimizer (경사 하강법) ──────────────────────
# learning_rate = 0.01 : 한 번에 얼마나 이동할지
train = tf.train.GradientDescentOptimizer(
    learning_rate=0.01
).minimize(cost)

print("Graph 설계 완료:")
print(f"  hypothesis: {hypothesis}")
print(f"  cost:       {cost}")
print(f"  train:      {train}")
print()
print("✓ 모두 Tensor 객체 — Session에서 실행하기 전까지는 숫자가 없음")


---
## 4. Session 실행 — 실제 학습

Graph를 실행해서 W와 b를 학습합니다.

### 경사 하강법 (Gradient Descent) 작동 원리
```
1. 현재 W, b로 hypothesis 계산
2. cost 계산 (예측 - 실제)²의 평균
3. cost가 줄어드는 방향으로 W, b를 조금씩 이동
4. 반복 → W → 1.0,  b → 0.0 으로 수렴
```


In [ ]:
# 학습 과정 기록용
history = {'step': [], 'cost': [], 'W': [], 'b': []}

with tf.Session() as sess:
    # ① 변수 초기화 (W, b에 난수 값 할당)
    sess.run(tf.global_variables_initializer())
    
    print(f"초기 W: {sess.run(W)[0]:.4f},  b: {sess.run(b)[0]:.4f}")
    print("-" * 50)
    
    # ② 학습 루프
    for step in range(2001):
        
        # train 실행 = Gradient Descent 1회
        _, cost_val, W_val, b_val = sess.run(
            [train, cost, W, b]
        )
        
        # 기록
        if step % 10 == 0:
            history['step'].append(step)
            history['cost'].append(cost_val)
            history['W'].append(W_val[0])
            history['b'].append(b_val[0])
        
        # 출력
        if step % 200 == 0:
            print(f"Step {step:4d} | cost: {cost_val:.6f} | W: {W_val[0]:.4f} | b: {b_val[0]:.4f}")

print("-" * 50)
print(f"\n최종 결과:")
print(f"  W = {history['W'][-1]:.6f}  (정답: 1.0)")
print(f"  b = {history['b'][-1]:.6f}  (정답: 0.0)")


---
## 5. 학습 과정 시각화


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Linear Regression 학습 과정', fontsize=14, fontweight='bold')

# ── 그래프 1: Cost 감소 곡선 ─────────────────────
axes[0].plot(history['step'], history['cost'], color='#E74C3C', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cost (MSE)')
axes[0].set_title('Cost 감소 곡선')
axes[0].set_yscale('log')   # 로그 스케일로 보기 쉽게
axes[0].grid(alpha=0.3)
axes[0].annotate(f"최종: {history['cost'][-1]:.2e}",
                 xy=(history['step'][-1], history['cost'][-1]),
                 xytext=(1000, history['cost'][5]),
                 fontsize=9, color='#E74C3C',
                 arrowprops=dict(arrowstyle='->', color='#E74C3C'))

# ── 그래프 2: W 수렴 과정 ────────────────────────
axes[1].plot(history['step'], history['W'], color='#3498DB', linewidth=2)
axes[1].axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='정답 W=1.0')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('W 값')
axes[1].set_title('W 수렴 과정')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].annotate(f"최종: {history['W'][-1]:.4f}",
                 xy=(history['step'][-1], history['W'][-1]),
                 xytext=(1000, history['W'][5]),
                 fontsize=9, color='#3498DB')

# ── 그래프 3: 학습된 직선 ────────────────────────
x_plot = np.linspace(0, 4, 100)
W_final = history['W'][-1]
b_final = history['b'][-1]

axes[2].scatter([1,2,3], [1,2,3], s=100, color='#E74C3C', zorder=5, label='데이터')
axes[2].plot(x_plot, W_final * x_plot + b_final,
             color='#3498DB', linewidth=2,
             label=f'학습 결과: y={W_final:.3f}x+{b_final:.3f}')
axes[2].plot(x_plot, x_plot, color='black', linestyle='--',
             alpha=0.4, label='정답: y=x')
axes[2].set_xlabel('x')
axes[2].set_ylabel('y')
axes[2].set_title('학습된 직선 vs 정답')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('linear_regression_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("그림 저장: linear_regression_result.png")


---
## 6. TensorFlow 핵심 개념 정리

### Graph vs Session
```python
# Graph 단계 (설계)
W = tf.Variable(...)          # 변수 선언만
cost = tf.reduce_mean(...)    # 연산 정의만 — 계산 안 함!

# Session 단계 (실행)
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())  # 변수 초기화
    result = sess.run(cost)                      # 실제 계산
```

### sess.run()의 반환값
```python
_, cost_val, W_val, b_val = sess.run([train, cost, W, b])
# train → None (업데이트만 수행, 반환값 없음)
# cost  → float 숫자
# W, b  → numpy array
```

### 경사 하강법 수식
$$W_{new} = W_{old} - \alpha \cdot \frac{\partial \text{cost}}{\partial W}$$

- $\alpha$ = learning_rate (0.01)
- $\frac{\partial \text{cost}}{\partial W}$ = cost의 기울기 (TF가 자동 계산)


---
## 7. ✏️ 실습 — 직접 바꿔보기


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 실습 1: learning_rate를 바꿔보세요
#   - 0.1  로 바꾸면? (너무 크면 발산)
#   - 0.001로 바꾸면? (너무 작으면 느림)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

LEARNING_RATE = 0.01   # ← 여기를 바꿔보세요

tf.set_random_seed(777)
x_train = [1, 2, 3]
y_train = [1, 2, 3]

W2 = tf.Variable(tf.random_normal([1]), name="weight2")
b2 = tf.Variable(tf.random_normal([1]), name="bias2")

hypothesis2 = x_train * W2 + b2
cost2 = tf.reduce_mean(tf.square(hypothesis2 - y_train))
train2 = tf.train.GradientDescentOptimizer(
    learning_rate=LEARNING_RATE
).minimize(cost2)

costs2 = []
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    for step in range(500):
        _, c = sess.run([train2, cost2])
        costs2.append(c)
    W_final = sess.run(W2)[0]
    b_final = sess.run(b2)[0]

plt.figure(figsize=(8, 4))
plt.plot(costs2, color='#E74C3C', linewidth=2)
plt.xlabel('Step'); plt.ylabel('Cost')
plt.title(f'learning_rate = {LEARNING_RATE}')
plt.yscale('log'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"최종 W={W_final:.4f}, b={b_final:.4f}")


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 실습 2: 데이터를 바꿔보세요
#   y = 2x + 1 이 되도록 — W=2, b=1 을 찾을까요?
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

x_train2 = [1, 2, 3, 4, 5]
y_train2 = [3, 5, 7, 9, 11]   # y = 2x + 1
# 정답: W = 2.0,  b = 1.0

# 여기에 코드 작성 ↓
# W3 = tf.Variable(...)
# b3 = tf.Variable(...)
# hypothesis3 = ...
# cost3 = ...
# train3 = tf.train.GradientDescentOptimizer(0.01).minimize(cost3)

# with tf.Session() as sess:
#     ...
#     print(f"W={..:.4f}, b={..:.4f}")


---
## 8. 📚 정리

| 구성 요소 | 코드 | 역할 |
|----------|------|------|
| `tf.Variable` | `W = tf.Variable(tf.random_normal([1]))` | 학습 파라미터 |
| Hypothesis | `hypothesis = x * W + b` | 예측 모델 |
| Cost | `tf.reduce_mean(tf.square(H - y))` | 오차 측정 |
| Optimizer | `GradientDescentOptimizer(lr).minimize(cost)` | 파라미터 업데이트 |
| Session | `sess.run(train)` | 실제 계산 실행 |

### 왜 TF1 스타일을 배우나?
- **Graph 개념** → 딥러닝 프레임워크의 본질적 동작 방식
- **자동 미분** → `minimize(cost)`가 내부적으로 편미분 계산
- **TF2와의 연결** → `tf.function`, `@tf.function`이 같은 Graph 개념

### 다음 단계
- `Lab 02-2`: 다중 입력 선형 회귀 (Matrix 연산)
- `Lab 03`: Logistic Regression (분류 문제)
- `Lab 04`: Multi-layer Perceptron
